# Challenge 5: Quantum Treasure Hunt ★★★

**The mission:** Treasure is hidden behind one of four doors, labeled `00`, `01`, `10`, and `11`. You have a magic **oracle** that can check doors for you. Searching the normal way, you might need 3 checks to be sure. **Grover's search algorithm** finds the treasure with **one** check, every time. Build it, then scale it up.

**Quantum ideas:** superposition, phase, interference.
**Best for:** CS and math folks, teams that want the hardest challenge, or teams that finished another one early.
**Time plan:** Missions 1 to 5 take about 90 minutes. Spend the rest on "Make it yours."

**Why it matters:** Grover's algorithm searches N items in about √N steps instead of N. For a million items, that's under a thousand steps instead of up to a million.

## The recipe

1. **Spread out:** put every door in equal superposition.
2. **Oracle:** secretly flip the sign (the phase) of the treasure door. Measuring right after this would show no change.
3. **Diffuser:** an interference step that boosts the marked door and cancels the others.
4. **Measure.**

In [ ]:
# Using Google Colab? Delete the "#" at the start of the next line, then run this cell once.
# !pip install -q "qiskit[visualization]"

In [ ]:
# SETUP: run this cell first (click it, then press Shift+Enter).
# You don't need to read or change anything in this cell.

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import math, random, qiskit

_sampler = StatevectorSampler()

def _needs_measure(qc):
    if qc.num_clbits == 0:
        raise ValueError("Your circuit has no measurements yet. Add qc.measure_all() as the last step of your circuit.")

def run(qc, shots=1000):
    """Run a circuit many times. Returns counts, e.g. {'0': 507, '1': 493}."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=shots).result()[0].join_data().get_counts()

def run_once(qc):
    """Run a circuit one time. Returns a single result as a string, e.g. '101'."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=1).result()[0].join_data().get_bitstrings()[0]

def run_many(qc, shots):
    """Run a circuit `shots` times. Returns every result in order, e.g. ['0', '1', '1', ...]."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=shots).result()[0].join_data().get_bitstrings()

def probabilities(qc):
    """Exact chance of each result, with no randomness. Measurements at the end are ignored."""
    sv = Statevector(qc.remove_final_measurements(inplace=False))
    return {str(k): round(float(v), 4) for k, v in sv.probabilities_dict().items() if v > 1e-9}

def _close(got, want, tol=0.02):
    return all(abs(got.get(k, 0) - want.get(k, 0)) <= tol for k in set(got) | set(want))

def _report(ok, yes, hint):
    if ok:
        print("\u2705 " + yes)
    else:
        print("\u274c Not yet. Hint: " + hint)

def _check_probs(qc, want, yes, hint):
    try:
        ok = _close(probabilities(qc), want)
    except Exception:
        ok = False
    _report(ok, yes, hint)

# ---- checks for this notebook ----
def _good_oracle(qc, door):
    zeros = [q for q in range(2) if door[::-1][q] == "0"]
    if zeros: qc.x(zeros)
    qc.cz(0, 1)
    if zeros: qc.x(zeros)

def _good_diffuser(qc):
    qc.h([0, 1]); qc.x([0, 1]); qc.cz(0, 1); qc.x([0, 1]); qc.h([0, 1])

def check_mission_1(spread_out):
    try:
        qc = QuantumCircuit(2); spread_out(qc)
        ok = _close(probabilities(qc), {"00": .25, "01": .25, "10": .25, "11": .25})
    except Exception:
        ok = False
    _report(ok, "Mission 1 complete: all four doors in superposition.", "one H gate on each qubit.")

def check_mission_2(oracle_11):
    try:
        qc = QuantumCircuit(2); qc.h([0, 1]); oracle_11(qc)
        ok = Statevector(qc).equiv(Statevector([0.5, 0.5, 0.5, -0.5]))
    except Exception:
        ok = False
    _report(ok, "Mission 2 complete: door 11 is secretly marked with a minus sign.", "a single qc.cz(0, 1).")

def check_mission_3(diffuser):
    try:
        qc = QuantumCircuit(2); qc.h([0, 1]); _good_oracle(qc, "11"); diffuser(qc)
        ok = _close(probabilities(qc), {"11": 1.0})
    except Exception:
        ok = False
    _report(ok, "Mission 3 complete: Grover finds door 11 every time!",
            "follow the five steps in order: H on both, X on both, CZ, X on both, H on both.")

def check_mission_4(oracle):
    try:
        ok = True
        for door in ["00", "01", "10", "11"]:
            qc = QuantumCircuit(2); qc.h([0, 1]); oracle(qc, door); _good_diffuser(qc)
            ok = ok and _close(probabilities(qc), {door: 1.0})
    except Exception:
        ok = False
    _report(ok, "Mission 4 complete: your oracle can mark any door.",
            "X on each qubit whose bit is '0' (bits[q] == '0'), then CZ, then the same X gates again.")

def check_mission_5(found, secret_door):
    _report(found == secret_door, "Mission 5 complete: treasure found in one oracle check!",
            "call run_once() on the circuit that grover(secret_door) builds.")

print(f"\u2705 Setup complete (Qiskit {qiskit.__version__}). You're ready to go!")

## Mission 1: Spread out

**TODO:** Put both qubits in superposition so every door has a 25% chance. Tip: `qc.h([0, 1])` applies H to both qubits at once.

In [ ]:
def spread_out(qc):
    # TODO: put both qubits in superposition
    pass

qc = QuantumCircuit(2)
spread_out(qc)
print(probabilities(qc))

In [ ]:
check_mission_1(spread_out)

## Mission 2: An oracle for door 11

The **CZ gate**, `qc.cz(0, 1)`, flips the sign of `11` and leaves the other doors alone. That's exactly an oracle that marks door `11`.

**TODO:** Mark door `11`.

In [ ]:
def oracle_11(qc):
    # TODO: mark door 11 with a CZ gate
    pass

qc = QuantumCircuit(2)
spread_out(qc)
oracle_11(qc)
print("Odds:", probabilities(qc))
Statevector(qc).draw("latex")

In [ ]:
check_mission_2(oracle_11)

The odds didn't change at all. The oracle only put a **minus sign** in front of $|11\rangle$ (look for it in the printout). Measuring can't see a sign, so the treasure is marked but still hidden. The diffuser's job is to turn that invisible sign into something you can measure.

## Mission 3: The diffuser

The two-qubit diffuser is five steps: **H on both, X on both, CZ, X on both, H on both.** It flips every amplitude around the average, which boosts the one with the minus sign. (If that sentence makes no sense yet, see "Watch the amplitudes" in Make it yours.)

**TODO:** Build the diffuser.

In [ ]:
def diffuser(qc):
    # TODO: H on both, X on both, CZ, X on both, H on both
    pass

qc = QuantumCircuit(2)
spread_out(qc)
oracle_11(qc)
diffuser(qc)
qc.measure_all()
print(run(qc))
qc.draw("mpl")

In [ ]:
check_mission_3(diffuser)

100% door `11`, with a single oracle check.

## Mission 4: An oracle for any door

CZ only marks `11`. To mark a different door, temporarily turn that door into `11` with X gates, apply CZ, then undo the X gates. The rule: **put an X on every qubit whose bit is 0** in the door's label.

Remember that Qiskit reads right to left. The code reverses the label for you, so `bits[0]` belongs to qubit 0 and `bits[1]` belongs to qubit 1.

**TODO:** Finish the oracle so it works for every door.

In [ ]:
def oracle(qc, door):
    bits = door[::-1]      # reverse, so bits[0] is qubit 0 and bits[1] is qubit 1
    # TODO: X on each qubit whose bit is "0", then CZ, then the same X gates again
    qc.cz(0, 1)   # <- only works for door '11' so far

def grover(door):
    qc = QuantumCircuit(2)
    spread_out(qc)
    oracle(qc, door)
    diffuser(qc)
    qc.measure_all()
    return qc

for door in ["00", "01", "10", "11"]:
    print("Treasure behind", door, "-> Grover finds", run(grover(door)))

In [ ]:
check_mission_4(oracle)

## Mission 5: The real treasure hunt

A quantum coin hides the treasure, and Grover gets **one** run to find it.

In real problems, the oracle is a circuit that can *check* an answer, like testing a password, without knowing where the answer is. Here we build the oracle from the secret door, but Grover still consults it only once.

**TODO:** Run Grover a single time and store its answer in `found`.

In [ ]:
picker = QuantumCircuit(2)
picker.h([0, 1])
picker.measure_all()
secret_door = run_once(picker)    # no peeking!

# TODO: run Grover ONCE and store the door it finds in `found`
found = "??"

print("Grover says the treasure is behind door", found)
print("The secret door was", secret_door)

In [ ]:
check_mission_5(found, secret_door)

## Make it yours

Pick an idea below or invent your own:

- **Eight doors (3 qubits).** Use `qc.ccz(0, 1, 2)` in place of `qc.cz(0, 1)` in both the oracle and the diffuser. One round of oracle + diffuser isn't enough anymore. Try 1, 2, and 3 rounds. What happens at 3? (Grover can "overcook.")
- **Success graph.** Plot the chance of finding the treasure against the number of rounds for 3 or 4 qubits. For 4 qubits, use `from qiskit.circuit.library import ZGate` and `qc.append(ZGate().control(3), [0, 1, 2, 3])` as the controlled-Z.
- **Two treasures.** Mark two doors at once. How many rounds do you need now?
- **Watch the amplitudes.** After each step, get `Statevector(qc).data.real` and draw it as a bar chart. Amplitudes can go negative! Show how the marked bar grows. Great for the Best Quantum Explainer award.
- **Race a classical searcher.** Over 100 random games, compare how many checks a classical guesser needs with Grover.

## Explain it

Answer in your own words and be ready to say them out loud.

1. Why does measuring right after the oracle show no change?
2. What does the diffuser do to the marked door?
3. Why does Grover need fewer checks than searching door by door?

Our answers: